# AuralScript-Lite — Section-by-Section Audio Analysis

**A browser-based tool for analyzing the spectral structure of songs.**

Built for Suno producers, cover artists, and anyone doing reference-track-driven music work. No installs required — Google Colab runs everything in the cloud for free.

---

## How to use this

1. Click **Runtime → Run all** in the menu above. Or run each cell with Shift+Enter.
2. When prompted, upload your mp3 (the song you want to analyze).
3. Read the section-by-section table and the energy curve.
4. Optionally: define custom sections that match the song's actual structure (intro / build / climax / etc.) and re-run for a more useful analysis.

**What you'll learn about your song:**
- Global BPM
- Spectral centroid (where the sound's energy sits — bright vs dark)
- RMS energy (loudness) per section
- Bass vs midrange balance per section
- Onset density (how busy each section is)
- Where the actual peak is (often surprising)

**Good for:**
- Diffing a Suno output against the source you're covering
- Finding the structural moves in a reference track (filter openings, bass walls)
- Validating that your AI-generated music actually matches the energy arc you wanted

**Not good for:**
- Neural genre tagging (use the full AuralScript with librosa+PANNs for that)
- Lyrics transcription (use Whisper directly)
- Sparse-drum or non-4/4 BPM detection (the autocorrelation approach is fragile)

---

Lineage: AuralScript-Lite was built collaboratively in a Claude conversation with Jason Alvey (Spankie0001), derived from the full AuralScript at https://github.com/Spankie0001/auralscript. CC0 / public domain. Use it however you want.

## Step 1 — Install ffmpeg (one line)

Colab already has numpy and scipy. We just need ffmpeg to decode mp3 files. This takes about 5 seconds.

In [ ]:
!apt-get install -y ffmpeg > /dev/null 2>&1 && echo '✓ ffmpeg ready'
import numpy, scipy, os, json
print(f'✓ numpy {numpy.__version__}')
print(f'✓ scipy {scipy.__version__}')

## Step 2 — Upload your mp3

Run the cell below. A file picker will appear. Choose your mp3. Keep it under 25 MB if possible (a 6-minute 128 kbps file is about 5 MB — well under the limit).

**Tip:** For your own Suno songs, download the mp3 from Suno's interface. For commercial tracks you're analyzing as a reference, use a song you own.

In [ ]:
from google.colab import files

uploaded = files.upload()
if not uploaded:
    raise RuntimeError('No file uploaded. Re-run this cell.')

SOURCE_MP3 = list(uploaded.keys())[0]
print(f'\n✓ Uploaded: {SOURCE_MP3} ({os.path.getsize(SOURCE_MP3)/1024/1024:.2f} MB)')

## Step 3 — Convert to mono WAV

ffmpeg downsamples to 22050 Hz mono. That's plenty for spectral analysis and keeps things fast.

In [ ]:
SOURCE_WAV = 'source.wav'
!ffmpeg -i "$SOURCE_MP3" -ac 1 -ar 22050 -y "$SOURCE_WAV" -loglevel error
print(f'✓ Converted: {SOURCE_WAV} ({os.path.getsize(SOURCE_WAV)/1024/1024:.2f} MB)')

## Step 4 — Define the analysis tool

This is the actual AuralScript-Lite code. You don't need to read it unless you're curious — just run the cell.

In [ ]:
import numpy as np
import scipy.io.wavfile as wav
from scipy import signal


def load_audio(wav_path):
    sr, audio = wav.read(wav_path)
    audio = audio.astype(np.float32) / 32768.0
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    return sr, audio


def analyze_section(audio, sr, start_s, end_s):
    start = int(start_s * sr)
    end = int(min(end_s * sr, len(audio)))
    seg = audio[start:end]
    if len(seg) < sr:
        return None

    rms = np.sqrt(np.mean(seg ** 2))
    rms_db = 20 * np.log10(rms + 1e-9)
    peak = np.max(np.abs(seg))
    peak_db = 20 * np.log10(peak + 1e-9)

    frame_len = int(0.05 * sr)
    hop = frame_len // 2
    centroids, bandwidths, flatness_list = [], [], []
    for i in range(0, len(seg) - frame_len, hop):
        frame = seg[i:i + frame_len] * np.hanning(frame_len)
        spec = np.abs(np.fft.rfft(frame))
        freqs = np.fft.rfftfreq(frame_len, 1 / sr)
        if np.sum(spec) > 0:
            cent = np.sum(freqs * spec) / np.sum(spec)
            centroids.append(cent)
            bw = np.sqrt(np.sum(((freqs - cent) ** 2) * spec) / np.sum(spec))
            bandwidths.append(bw)
            spec_pos = spec[spec > 1e-10]
            if len(spec_pos) > 10:
                geo = np.exp(np.mean(np.log(spec_pos + 1e-10)))
                arith = np.mean(spec_pos)
                if arith > 0:
                    flatness_list.append(geo / arith)

    centroid_mean = float(np.mean(centroids)) if centroids else 0.0
    bandwidth_mean = float(np.mean(bandwidths)) if bandwidths else 0.0
    flatness_mean = float(np.mean(flatness_list)) if flatness_list else 0.0

    sos_bass = signal.butter(4, [20, 250], btype='bandpass', fs=sr, output='sos')
    sos_mid = signal.butter(4, [1000, 4000], btype='bandpass', fs=sr, output='sos')
    sos_hi = signal.butter(4, [4000, 10000], btype='bandpass', fs=sr, output='sos')
    bass = signal.sosfilt(sos_bass, seg)
    mid = signal.sosfilt(sos_mid, seg)
    hi = signal.sosfilt(sos_hi, seg)
    bass_db = 20 * np.log10(np.sqrt(np.mean(bass ** 2)) + 1e-9)
    mid_db = 20 * np.log10(np.sqrt(np.mean(mid ** 2)) + 1e-9)
    hi_db = 20 * np.log10(np.sqrt(np.mean(hi ** 2)) + 1e-9)

    fl = int(0.023 * sr)
    frame_energies = np.array([
        np.sum(seg[i:i + fl] ** 2) for i in range(0, len(seg) - fl, fl)
    ])
    if len(frame_energies) > 1:
        diffs = np.diff(frame_energies)
        diffs[diffs < 0] = 0
        threshold = np.mean(diffs) + 1.5 * np.std(diffs)
        onsets = int(np.sum(diffs > threshold))
        onsets_per_sec = onsets / (end_s - start_s)
    else:
        onsets_per_sec = 0.0

    return {
        'rms_db': float(rms_db),
        'peak_db': float(peak_db),
        'centroid': centroid_mean,
        'bandwidth': bandwidth_mean,
        'flatness': flatness_mean,
        'bass_db': float(bass_db),
        'mid_db': float(mid_db),
        'hi_db': float(hi_db),
        'bass_vs_mid': float(bass_db - mid_db),
        'onsets_per_sec': float(onsets_per_sec),
    }


def estimate_bpm(audio, sr):
    hop = 512
    n_frames = len(audio) // hop
    env = np.zeros(n_frames)
    for i in range(n_frames - 1):
        f1 = np.abs(np.fft.rfft(audio[i * hop:(i + 1) * hop] * np.hanning(hop)))
        f2 = np.abs(np.fft.rfft(audio[(i + 1) * hop:(i + 2) * hop] * np.hanning(hop)))
        diff = f2 - f1
        diff[diff < 0] = 0
        env[i] = np.sum(diff)
    env = env - np.mean(env)
    ac = np.correlate(env, env, mode='full')
    ac = ac[len(ac) // 2:]
    frames_per_sec = sr / hop
    min_lag = int(60 / 200 * frames_per_sec)
    max_lag = int(60 / 60 * frames_per_sec)
    ac_section = ac[min_lag:max_lag]
    if len(ac_section) > 0:
        peak_lag = np.argmax(ac_section) + min_lag
        return float(60 / (peak_lag / frames_per_sec))
    return 0.0


def compute_curve(audio, sr, window_s=5.0, hop_s=1.0):
    win = int(window_s * sr)
    hop = int(hop_s * sr)
    rms_envelope, centroid_envelope, times = [], [], []
    for i in range(0, len(audio) - win, hop):
        seg = audio[i:i + win]
        rms = np.sqrt(np.mean(seg ** 2))
        rms_envelope.append(20 * np.log10(rms + 1e-9))
        spec = np.abs(np.fft.rfft(seg * np.hanning(len(seg))))
        freqs = np.fft.rfftfreq(len(seg), 1 / sr)
        if np.sum(spec) > 0:
            cent = np.sum(freqs * spec) / np.sum(spec)
            centroid_envelope.append(cent)
        else:
            centroid_envelope.append(0)
        times.append(i / sr)
    return np.array(times), np.array(rms_envelope), np.array(centroid_envelope)


print('✓ AuralScript-Lite loaded.')

## Step 5 — Run analysis with 8 equal sections (default view)

This gives you a quick overview. The 8 sections are arbitrary equal-width chunks. Useful for spotting the shape of the song before you decide on real section boundaries.

In [ ]:
sr, audio = load_audio(SOURCE_WAV)
duration = len(audio) / sr

print(f'Duration: {duration:.1f}s ({int(duration//60)}:{int(duration%60):02d})')
print(f'Sample rate: {sr} Hz')
print()

bpm = estimate_bpm(audio, sr)
print(f'Estimated global BPM: {bpm:.1f}')
print()

n = 8
width = duration / n
default_sections = [
    {'name': f'SEG_{i+1:02d}', 'start': i*width, 'end': (i+1)*width}
    for i in range(n)
]

print(f"{'SECTION':<10}{'TIME':<14}{'RMS':<10}{'CENT':<10}{'BW':<8}{'FLAT':<8}{'BASS-MID':<11}{'ONS/s':<7}")
print('-' * 80)
for sec in default_sections:
    r = analyze_section(audio, sr, sec['start'], sec['end'])
    if r is None:
        continue
    print(f"{sec['name']:<10}{sec['start']:>4.0f}-{sec['end']:<8.0f} "
          f"{r['rms_db']:>6.1f}    {r['centroid']:>6.0f}    "
          f"{r['bandwidth']:>5.0f}   {r['flatness']:>5.3f}   "
          f"{r['bass_vs_mid']:>+7.1f}    {r['onsets_per_sec']:>5.1f}")

## Step 6 — Print the whole-track energy curve

This is the most useful output for finding structural moves. The bar chart shows energy over time. Look for:

- **Where the energy peak is** (often not where you'd guess)
- **Where centroid jumps suddenly** (filter opening events)
- **Where centroid drops while energy stays high** (the "bass wall" producer move)
- **Where it all collapses** (the burnup/outro)

Write down the timestamps of any structural transitions you see. You'll use them in Step 7.

In [ ]:
times, rms_db, centroid_hz = compute_curve(audio, sr)

print(f"{'TIME':<8}{'RMS(dB)':<10}{'CENTROID(Hz)':<14}{'ENERGY':<35}")
print('-' * 70)
rms_min, rms_max = rms_db.min(), rms_db.max()
rng = max(rms_max - rms_min, 0.001)
step = 5
for i in range(0, len(times), step):
    norm = (rms_db[i] - rms_min) / rng
    bar = '█' * int(norm * 30)
    mm = int(times[i] // 60)
    ss = int(times[i] % 60)
    print(f"  {mm:>2}:{ss:02d}  {rms_db[i]:>5.1f}     {centroid_hz[i]:>6.0f}        {bar}")

peak_idx = int(np.argmax(rms_db))
peak_c_idx = int(np.argmax(centroid_hz))
print()
print(f'PEAK ENERGY:    {int(times[peak_idx]//60)}:{int(times[peak_idx]%60):02d}  '
      f'({rms_db[peak_idx]:.1f} dB)')
print(f'PEAK CENTROID:  {int(times[peak_c_idx]//60)}:{int(times[peak_c_idx]%60):02d}  '
      f'({centroid_hz[peak_c_idx]:.0f} Hz — often this is the noise/burnup)')

## Step 7 — Define your own sections (the useful part)

Now that you know the song's shape, edit the cell below with your own section boundaries. Match the actual musical structure: intro / verse / build / drop / climax / outro. Use the timestamps you noted from Step 6.

Edit the `MY_SECTIONS` list below, then run the cell.

In [ ]:
# Edit this list to match your song's actual structure.
# Format: {'name': 'LABEL', 'start': seconds, 'end': seconds, 'note': 'description'}

MY_SECTIONS = [
    {'name': 'INTRO',     'start': 0.0,   'end': 25.0,  'note': 'spoken word + pad'},
    {'name': 'ARP_IN',    'start': 25.0,  'end': 50.0,  'note': 'arpeggio enters'},
    {'name': 'DRUMS_IN',  'start': 50.0,  'end': 100.0, 'note': 'drums establish'},
    {'name': 'BUILD',     'start': 100.0, 'end': 170.0, 'note': 'filter opening'},
    {'name': 'CLIMAX_1',  'start': 170.0, 'end': 230.0, 'note': 'first peak'},
    {'name': 'CLIMAX_2',  'start': 230.0, 'end': 290.0, 'note': 'sustained wall'},
    {'name': 'OUTRO',     'start': 290.0, 'end': 320.0, 'note': 'collapse / fade'},
]

print(f"{'SECTION':<12}{'TIME':<14}{'RMS':<10}{'CENT':<10}{'BW':<8}{'FLAT':<8}{'BASS-MID':<11}{'ONS/s':<7}")
print('-' * 82)
results = {}
for sec in MY_SECTIONS:
    r = analyze_section(audio, sr, sec['start'], sec['end'])
    if r is None:
        print(f"{sec['name']:<12}{sec['start']:>4.0f}-{sec['end']:<8.0f}  (too short or out of bounds)")
        continue
    results[sec['name']] = r
    print(f"{sec['name']:<12}{sec['start']:>4.0f}-{sec['end']:<8.0f} "
          f"{r['rms_db']:>6.1f}    {r['centroid']:>6.0f}    "
          f"{r['bandwidth']:>5.0f}   {r['flatness']:>5.3f}   "
          f"{r['bass_vs_mid']:>+7.1f}    {r['onsets_per_sec']:>5.1f}")

print()
print('Section notes:')
for sec in MY_SECTIONS:
    if sec.get('note'):
        print(f"  {sec['name']:<12}{sec['note']}")

## Step 8 — How to read these numbers

**RMS (dB)** — How loud the section is. -25 is loud. -40 is quiet. The difference between intro and climax is usually 10-15 dB.

**CENT (Hz)** — Spectral centroid. Where the "brightness" of the sound lives.
- 800-1500 Hz = dark, warm, bass-and-mids dominant
- 1500-2500 Hz = neutral / midrange-led
- 2500-3500 Hz = bright, hi-mid forward
- 3500+ Hz = very bright, often noise/distortion territory

**BW (Hz)** — Bandwidth. How spread the sound is around the centroid. Higher = wider spectrum (more like full-band music). Lower = narrower (like a single instrument).

**FLAT** — Spectral flatness. 0 = pure tone. 1 = white noise. >0.4 = distorted / noisy / many overtones.

**BASS-MID** — Bass band energy (20-250 Hz) minus midrange band energy (1-4 kHz).
- Positive = bass-heavy mix
- 0 = balanced
- Negative = mid/treble-heavy (vocals, acoustic guitar, etc.)

**ONS/s** — Onsets per second. How busy the section is.
- 0-1 = sparse (pads, ambient)
- 1-3 = moderate (typical pop/rock)
- 3+ = busy (techno, dense arrangements)

---

## How to use this for Suno cover work

1. Run the source track through this notebook. Note the curve and the section table.
2. Identify 2-3 distinctive structural moves: a filter opening, a bass wall, a noise collapse, etc.
3. Write your Suno style prompt around those moves explicitly. Use the centroid and BASS-MID values as targets in your prompt language.
4. Generate.
5. Download your Suno output and run it through this notebook too.
6. Compare section by section. Wherever the values diverge significantly, that's where your prompt didn't land.
7. Adjust the prompt for that specific section and regenerate.

This is the closed-loop methodology that makes generative cover work tractable. Without it, you're guessing.

---

## More

Full AuralScript (with PANNs neural tagging and Whisper transcription): https://github.com/Spankie0001/auralscript

Questions or improvements: open an issue on the repo above.